# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed on hosts with an old system SQLite

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.tooling import tool

In [3]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
chroma_client = chromadb.PersistentClient(path="chromadb")
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(query_texts=[query], n_results=3)
    return results["metadatas"][0]

#### Evaluate Retrieval Tool

In [5]:
class EvaluationReport(BaseModel):
    useful: bool
    description: str


@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(model="gpt-4o-mini", temperature=0)
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n\n"
        f"Query: {question}\n\n"
        f"Documents: {retrieved_docs}"
    )
    response = llm.invoke(prompt, response_format=EvaluationReport)
    return EvaluationReport.model_validate_json(response.content).model_dump()

#### Game Web Search Tool

In [6]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    result = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        max_results=3,
    )
    return {
        "answer": result.get("answer", ""),
        "results": [
            {"title": r["title"], "url": r["url"], "content": r["content"]}
            for r in result.get("results", [])
        ],
    }

### Agent

In [7]:
agent = Agent(
    model_name="gpt-4o-mini",
    temperature=0,
    instructions=(
        "You are UdaPlay, a research assistant for the video game industry.\n"
        "For every question, follow these steps:\n"
        "1. Call retrieve_game to search the internal game database.\n"
        "2. Call evaluate_retrieval with the question and the retrieved documents.\n"
        "3. Only if the evaluation says the documents are not useful, call game_web_search.\n"
        "4. Answer using this format:\n"
        "Answer: <the answer>\n"
        "Source: <'Internal database' or the URL(s) of the web pages used>\n"
        "Confidence: <high, medium or low>\n"
        "Use the earlier conversation to resolve follow-up questions."
    ),
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
)


def ask(query, session_id="udaplay"):
    run = agent.invoke(query, session_id=session_id)
    messages = run.get_final_state()["messages"]
    # Print only this turn: everything after the last user message
    start = max(i for i, m in enumerate(messages) if m.role == "user")
    print(f"QUESTION: {query}\n")
    for m in messages[start + 1:]:
        if m.role == "assistant" and m.tool_calls:
            for call in m.tool_calls:
                print(f"TOOL CALL: {call.function.name}({call.function.arguments})")
        elif m.role == "tool":
            print(f"TOOL RESULT ({m.name}): {m.content[:500]}\n")
        elif m.role == "assistant":
            print(f"FINAL ANSWER:\n{m.content}")

In [8]:
ask("When was Pokémon Gold and Silver released?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: When was Pokémon Gold and Silver released?

TOOL CALL: retrieve_game({"query":"Pokémon Gold and Silver release date"})
TOOL RESULT (retrieve_game): "[{'YearOfRelease': 1999, 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay mechanics.', 'Name': 'Pok\u00e9mon Gold and Silver', 'Publisher': 'Nintendo', 'Platform': 'Game Boy Color', 'Genre': 'Role-playing'}, {'Genre': 'Role-playing', 'Description': 'Third-generation Pok\u00e9mon games set in the Hoenn region, featuring new Pok\u00e9mon and double battles.', 'Name': 'Pok\u00e9mon Ruby and Sapphire', 'Publisher': 'Nintendo', 'YearOfRelease':

TOOL CALL: evaluate_retrieval({"question":"When was Pokémon Gold and Silver released?","retrieved_docs":"[{'YearOfRelease': 1999, 'Description': 'Second-generation Pok\u0000mon games introducing new regions, Pok\u0000mon, and gameplay mechanics.', 'Na

In [9]:
ask("Which one was the first 3D platformer Mario game?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Which one was the first 3D platformer Mario game?

TOOL CALL: retrieve_game({"query":"first 3D platformer Mario game"})
TOOL RESULT (retrieve_game): "[{'Publisher': 'Nintendo', 'Genre': 'Platformer', 'Description': \"A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.\", 'YearOfRelease': 1996, 'Platform': 'Nintendo 64', 'Name': 'Super Mario 64'}, {'Genre': 'Platformer', 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', 'YearOfRelease': 1990, 'Name': 'Super Mario World', 'Platform': 'Super Nintendo E

TOOL CALL: evaluate_retrieval({"question":"Which one was the first 3D platformer Mario game?","retrieved_docs":"[{'Publisher': 'Nintendo', 'Genre': 'Platformer', 'Description': \"A groundbreaking 3D platformer that set new standards for the genre, fe

In [10]:
ask("Was Mortal Kombat X released for PlayStation 5?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Was Mortal Kombat X released for PlayStation 5?

TOOL CALL: retrieve_game({"query":"Mortal Kombat X PlayStation 5 release"})
TOOL RESULT (retrieve_game): "[{'Publisher': 'Sony Interactive Entertainment', 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.', 'Platform': 'PlayStation 5', 'Genre': 'Action-adventure', 'YearOfRelease': 2023, 'Name': \"Marvel's Spider-Man 2\"}, {'Publisher': 'Sony Interactive Entertainment', 'Platform': 'PlayStation 4', 'YearOfRelease': 2018, 'Genre': 'Action-adventure', 'Description': 'An open-world superhero game that lets players swing th

TOOL CALL: evaluate_retrieval({"question":"Was Mortal Kombat X released for PlayStation 5?","retrieved_docs":"[{'Publisher': 'Sony Interactive Entertainment', 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Park

### Conversation state

A follow-up in the same session uses the earlier questions.

In [11]:
ask("Who published the Pokémon game I asked about first?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Who published the Pokémon game I asked about first?

TOOL CALL: retrieve_game({"query":"Pokémon Gold and Silver publisher"})
TOOL RESULT (retrieve_game): "[{'Publisher': 'Nintendo', 'Genre': 'Role-playing', 'YearOfRelease': 1999, 'Name': 'Pok\u00e9mon Gold and Silver', 'Platform': 'Game Boy Color', 'Description': 'Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay mechanics.'}, {'Name': 'Pok\u00e9mon Ruby and Sapphire', 'Description': 'Third-generation Pok\u00e9mon games set in the Hoenn region, featuring new Pok\u00e9mon and double battles.', 'Publisher': 'Nintendo', 'YearOfRelease': 2002, 'Genre': 'Role-pla

TOOL CALL: evaluate_retrieval({"question":"Who published the Pokémon game I asked about first?","retrieved_docs":"[{'Publisher': 'Nintendo', 'Genre': 'Role-playing', 'YearOfRelease': 1999, 'Name': 'Pok\u0000mon Gold and Silver', 'Platform': 'Gam